# Notebook 02 — Multi-Agent Retrieval Orchestration

## Purpose

This notebook demonstrates the **multi-agent retrieval system** built on top of the `advanced-genai-rag` package.  
It covers the complete retrieval pipeline from query to ranked answer, walking through each component in detail.

## System Architecture

```
User Query
    │
    ▼
┌───────────────────────────────────────────────────────────┐
│                    Orchestration Layer                    │
│                                                           │
│  ┌───────────┐   ┌──────────────┐   ┌───────────────┐   │
│  │ Waterfall │   │    Voting    │   │  Confidence   │   │
│  │ Strategy  │   │   Strategy   │   │   Strategy    │   │
│  └─────┬─────┘   └──────┬───────┘   └───────┬───────┘   │
│        └────────────────┼────────────────────┘           │
│                         ▼                                 │
│              Weighted RRF Fusion                          │
└───────────────────┬───────────────────────────────────────┘
                    │
    ┌───────────────┼───────────────┐
    ▼               ▼               ▼
┌───────┐     ┌─────────┐     ┌─────────┐
│ BM25  │     │  Dense  │     │  Graph  │
│(+QE)  │     │  (E5)   │     │  RAG    │
└───────┘     └─────────┘     └─────────┘
    │               │               │
    └───────────────┼───────────────┘
                    ▼
            Document Corpus
          (EN + DE documents)
```

## What you will learn

1. How to build a **bilingual BM25 retriever** with pseudo-relevance feedback (PRF)
2. How **dense retrieval** works with multilingual embeddings (E5)
3. How **GraphRAG** traverses knowledge graphs for retrieval
4. How **Reciprocal Rank Fusion (RRF)** merges ranked lists from multiple retrievers
5. How three orchestration strategies differ:
   - **Waterfall**: starts lean, adds GraphRAG only when initial results are weak
   - **Voting**: always fuses all three retrievers with fixed weights
   - **Confidence**: dynamically adjusts weights based on query type classification
6. How to evaluate retrieval quality using standard IR metrics (P@k, MRR, NDCG)

## Prerequisites

Before running this notebook you need:
- A pre-built **BM25 pickle** (`bm25_fixed_qe.pkl`) from Step 1
- A **Chroma vector store** for dense retrieval (`vectordb_dense/fixed_e5/`)
- A **GraphRAG retriever module** (`load_graphrag.py`) from Step 1
- A **bilingual QA benchmark** (`benchmark_qa_bilingual.json`)
- **Relevance judgments** (qrels, one JSON per document)

All paths are configured in **Section 2** below.

## Section 1 — Installation

### Dependencies  
This notebook is opened through Google Colab.
To access the RAG and agents pipeline, you need to install its package from the GitHub repository.
Note that the pipeline has expressedly been build as a package for this purpose.


**NOTE: The current package is downloaded from the `refactor/baseline-package` branch.**

If this package has been moved to main in the meantime, use the following url: "git+https://github.com/dhub100/advanced-genai-rag.git@main#subdirectory=baseline/package/"

In [9]:
!pip install --quiet --upgrade pip setuptools wheel
!pip install "git+https://github.com/dhub100/advanced-genai-rag.git@refactor/baseline-package#subdirectory=baseline/package/"

  Cloning https://github.com/dhub100/advanced-genai-rag.git (to revision refactor/baseline-package) to /tmp/pip-req-build-obp5sba2
  Running command git clone --filter=blob:none --quiet https://github.com/dhub100/advanced-genai-rag.git /tmp/pip-req-build-obp5sba2
  Running command git checkout -b refactor/baseline-package --track origin/refactor/baseline-package
  Switched to a new branch 'refactor/baseline-package'
  Branch 'refactor/baseline-package' set up to track remote branch 'refactor/baseline-package' from 'origin'.
  Resolved https://github.com/dhub100/advanced-genai-rag.git to commit 638219c4e4f415d3465a8a986671ae24ca08bc44
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Next, we'll import the required dependencies. Everything should depend on the pipeline package for reproducibility

**Every external library should be added to the pipeline package directly.**

Key libraries:

- **`torch`** — GPU/CPU device management and model inference
- **`nltk`** — tokenisation and stopword lists for both English and German
- **`rank_bm25`** — fast BM25Okapi implementation
- **`langchain_*`** — document abstractions (`Document`), vector store wrapper (`Chroma`), and embedding models
- **`transformers`** — M2M100 translation model, flan-t5 classifier, Mistral answer synthesizer
- **`langdetect`** — lightweight language detection for automatic query routing
- **`pytrec_eval`** — standard TREC evaluation library (wraps the C implementation)
- **`pandas / tqdm`** — results display and progress reporting

In [2]:
# Standard library
import pickle
import os
import json
import time
import re
import functools
import importlib.machinery
from collections import defaultdict, Counter
from typing import List, Dict, Optional, Tuple

# External Library
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import nltk
from langdetect import detect
from rank_bm25 import BM25Okapi
from transformers import (
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    AutoModelForCausalLM,
)
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import pytrec_eval


### Data
The data used to run this notebook is heavy and is not stored on GitHub. It is stored in a shared GoogleDrive.
Run the following command to connect Google Colab to google drive, access the shared folder and then the data.

**Note:** The following line of code will open a popup window to sign-in to Google Drive.

The following code will mount Google Drive at the 'root'. Your personnal drive and shared drives are child directories of the mounted root. Such that:

* Personal Drive `MyDrive`: '/content/drive/MyDrive'
* Shared Drive `Shareddrives/Adv_GenAI_FS26/`: '/content/drive/Shareddrives/Adv_GenAI_FS26'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Section 2 — Configuration

All file paths and evaluation settings live here.  
Edit these variables to match your local storage layout before running the rest of the notebook.

| Variable | Description |
|---|---|
| `ROOT` | Root directory that contains `storage/`, `benchmark/` |
| `PATH_BM25_PICKLE` | Serialised `QEBM25` object from Step 1 |
| `PATH_DENSE_LOADER` | Python module that exposes `load_dense_fixed()` |
| `PATH_GRAG_LOADER` | Python module that exposes `retrieve()` |
| `PATH_QA` | Bilingual QA benchmark JSON |
| `PATH_QRELS_FIXED` | Folder of relevance-judgement JSON files (one per document) |
| `METRICS` | `pytrec_eval` metric names to compute |

In [ ]:
# ── Edit these paths to match your setup ─────────────────────────────────────
ROOT = pathlib.Path("/content/drive/Shareddrives/Adv_GenAI_FS26/")

PATH_BM25_PICKLE  = ROOT / "storage/subsample/retrieval_downstream/bm25_fixed_qe.pkl"
PATH_DENSE_LOADER = ROOT / "storage/subsample/vectordb_dense/load_dense_fixed.py"
PATH_GRAG_LOADER  = ROOT / "storage/subsample/retrieval_graph/load_graphrag.py"
PATH_QA           = ROOT / "benchmark/benchmark_qa_bilingual.json"
PATH_QRELS_FIXED  = ROOT / "benchmark/score/fixed_size"

# IR metrics to compute during evaluation
METRICS = {"P_5", "P_10", "recall_100", "recip_rank", "ndcg_cut_10"}

# Sanity-check that the critical paths exist
for label, p in [
    ("BM25 pickle", PATH_BM25_PICKLE),
    ("Dense loader", PATH_DENSE_LOADER),
    ("GraphRAG loader", PATH_GRAG_LOADER),
    ("QA benchmark", PATH_QA),
    ("Qrels folder", PATH_QRELS_FIXED),
]:
    status = "✓" if p.exists() else "✗ NOT FOUND"
    print(f"  {status}  {label}: {p}")

  ✗ NOT FOUND  BM25 pickle: /content/drive/Shareddrives/Adv_GenAI_FS26/storage/subsample/retrieval_downstream/bm25_fixed_qe.pkl
  ✗ NOT FOUND  Dense loader: /content/drive/Shareddrives/Adv_GenAI_FS26/storage/subsample/vectordb_dense/load_dense_fixed.py
  ✗ NOT FOUND  GraphRAG loader: /content/drive/Shareddrives/Adv_GenAI_FS26/storage/subsample/retrieval_graph/load_graphrag.py
  ✗ NOT FOUND  QA benchmark: /content/drive/Shareddrives/Adv_GenAI_FS26/benchmark/benchmark_qa_bilingual.json
  ✗ NOT FOUND  Qrels folder: /content/drive/Shareddrives/Adv_GenAI_FS26/benchmark/score/fixed_size


## Section 3 — Environment


In [ ]:
# ── NLTK data downloads (skip if already cached) ─────────────────────────────
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("punkt_tab", quiet=True)

STOP_EN = set(nltk.corpus.stopwords.words("english"))
STOP_DE = set(nltk.corpus.stopwords.words("german"))

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cpu


## Section 4 — Translation Component

### Why bilingual retrieval?

The ETH Zurich news corpus contains documents in both **English and German**.  
Users may ask questions in either language. Restricting retrieval to only the query language would miss relevant documents in the other language.

### Solution: EN↔DE Translation + Dual Retrieval

For every query we:
1. Detect its source language (English or German)
2. Translate it to the other language
3. Retrieve documents using **both** the original and translated query
4. Merge and deduplicate results by document ID

### Model: `facebook/m2m100_418M`

Facebook's M2M100 is a many-to-many multilingual translation model trained on 100 languages.  
The 418M parameter variant is used here for speed. Key properties:
- No English pivot: can translate directly between any pair of supported languages
- 128-token `max_new_tokens` is sufficient for typical IR queries
- `num_beams=1` (greedy decoding) keeps latency low — translation quality is secondary to coverage

### Caching

`functools.lru_cache` caches translation results by `(text, tgt)` key.  
Since the QA benchmark is small (≤25 questions), many queries are repeated across evaluation runs, so the cache pays off immediately.

> ⚠️ Note that the next code cells is just for description only. The actual function is implemented inside `rag.retrieval.translator`. The pipeline just import it through the package for simplicity.

In [ ]:
class EnDeTranslator:
    """Greedy EN⇄DE translation (cached – 1st call downloads weights)."""

    def __init__(self, model="facebook/m2m100_418M", device: str | None = None) -> None:
        self.device = device
        self.tok = M2M100Tokenizer.from_pretrained(model)
        self.model = M2M100ForConditionalGeneration.from_pretrained(model).to(
            self.device
        )

    @functools.lru_cache(maxsize=512)
    def translate(self, text: str, tgt: str) -> str:
        src = detect(text) if text.strip() else "en"
        src = src if src in ("en", "de") else "en"
        if src == tgt:
            return text
        self.tok.src_lang = src
        ids = self.tok(text, return_tensors="pt").to(self.device)
        out = self.model.generate(
            **ids,
            forced_bos_token_id=self.tok.get_lang_id(tgt),
            num_beams=1,
            max_new_tokens=128,
        )
        return self.tok.decode(out[0], skip_special_tokens=True)

In [4]:
from rag.retrieval.translator import EnDeTranslator

# Instantiate — first call downloads ~1.7 GB of weights
translator = EnDeTranslator()

# Quick smoke test
print(translator.translate("Who was president of ETH in 2003?", "de"))
print(translator.translate("Wer war 2003 Präsident der ETH?", "en"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Attempting to cast a BatchEncoding to type None. This is not supported.
The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Attempting to cast a BatchEncoding to type None. This is not supported.
Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Wer war 2003 Präsident von ETH?
Who was the President of the ETH in 2003?


## Section 5 — Query Expansion with Pseudo-Relevance Feedback

### Problem: vocabulary mismatch

BM25 is a **term-frequency** model. If a document uses synonyms or different phrasing of the same concept, it scores poorly even if it is highly relevant.  
For example, a query for `"ETH president 2003"` may miss documents that say `"ETH Zurich rector Kübler"` because `president` ≠ `rector`.

### Solution: Pseudo-Relevance Feedback (PRF)

PRF is a classic IR technique:
1. Retrieve the top-*k* documents using the original query
2. Assume those top-*k* documents are relevant (the "pseudo" part)
3. Extract the most frequent non-stopword tokens from their text
4. Append those tokens to the original query
5. Re-retrieve using the expanded query

This automatically adds terms like `rector`, `Kübler`, `ETH Zurich` to the query without requiring manual synonym lists.

### Parameters

| Parameter | Value | Meaning |
|---|---|---|
| `fb_docs` | 5 | Number of feedback documents to read |
| `fb_terms` | 5 | Number of expansion terms to add |

Using too many feedback documents or terms can introduce noise and hurt precision — these values are empirically tuned.



> ⚠️ Note that the following code is just for description only. The actual function is implemented inside `rag.retrieval.agents.bm25.QEBM25`.



In [ ]:
def _expand_query(query: str, base_retriever, fb_docs: int = 5, fb_terms: int = 5) -> str:
    """
    Pseudo-relevance feedback: retrieves top-`fb_docs` documents, then
    appends the `fb_terms` most frequent non-stopword tokens to the query.

    Example:
        query = "ETH president 2003"
        expanded → "ETH president 2003 kübler rector zurich professorship tenure"
    """
    hits = base_retriever.search(query, top_k=fb_docs)
    tokens = [
        t.lower()
        for h in hits
        for t in nltk.word_tokenize(h.page_content.lower())
        if t.isalpha() and t not in STOP_EN and t not in STOP_DE
    ]
    extra = " ".join(w for w, _ in nltk.FreqDist(tokens).most_common(fb_terms))
    return f"{query} {extra}" if extra else query

## Section 6 — Bilingual BM25 Retriever

### BM25 background

BM25 (Best Match 25) is a probabilistic ranking function that extends TF-IDF with:
- **Term saturation**: repeated occurrences of a term have diminishing returns (`k1` parameter)
- **Document length normalisation**: prevents long documents from dominating (`b` parameter)

Score formula for document `d` given query `q`:
```
score(d, q) = Σ IDF(t) * (tf(t,d) * (k1+1)) / (tf(t,d) + k1 * (1 - b + b * |d|/avgdl))
```

### `BilingualBM25` design

Two **separate** BM25 indices are maintained — one for English documents, one for German.  
When a query arrives:
1. Its language is detected
2. It is translated into the other language using `EnDeTranslator`
3. Both indices are searched independently
4. Results are **deduplicated** by `chunk_id` / `record_id`, keeping the higher score per document
5. The merged list is sorted by BM25 score and truncated to `top_k`

### `QEBM25` wrapper

Combines `BilingualBM25` with PRF via `_expand_query`. This is the retriever that is **serialised to disk** during Step 1 and loaded here via pickle.

> ⚠️ Note that the next code cells is just for description only. The actual function is implemented inside `rag.retrieval.agents.bm25`. The pipeline just import it through the package for simplicity.

In [ ]:
class BilingualBM25:
    """
    Dual BM25Okapi index — one index per language (EN, DE).

    For each query the class:
    1. Detects the query language
    2. Translates the query into the other language
    3. Searches both indices
    4. Deduplicates by chunk/record ID (keeping highest BM25 score)
    5. Returns results sorted by score
    """

    def __init__(self, docs: List[Document]) -> None:
        self.docs_by_lang = {"en": [], "de": []}
        self.toks_by_lang = {"en": [], "de": []}
        for d in docs:
            lang = d.metadata.get("language", "en")
            lang = lang if lang in ("en", "de") else "en"
            self.docs_by_lang[lang].append(d)
            self.toks_by_lang[lang].append(nltk.word_tokenize(d.page_content))
        # Build one BM25Okapi per language
        self.bm25 = {l: BM25Okapi(tok) for l, tok in self.toks_by_lang.items() if tok}

    def _rank_lang(self, q: str, lang: str, k: int) -> List[Document]:
        """Score all documents in `lang` against query `q` and return top-`k`."""
        scores = self.bm25[lang].get_scores(nltk.word_tokenize(q))
        idx = np.argsort(scores)[::-1][:k]
        hits = []
        for i in idx:
            d = self.docs_by_lang[lang][i]
            d.metadata["bm25_score"] = float(scores[i])
            hits.append(d)
        return hits

    def search(self, query: str, top_k: int = 100) -> List[Document]:
        """Search both EN and DE indices and return merged, deduplicated results."""
        src = detect(query) if query.strip() else "en"
        src = src if src in ("en", "de") else "en"
        bag = []
        for lang in ("en", "de"):
            q_lang = translator.translate(query, lang) if lang != src else query
            bag.extend(self._rank_lang(q_lang, lang, top_k))
        # Deduplicate: keep the entry with the highest BM25 score per document
        best: Dict[str, Document] = {}
        for d in bag:
            uid = d.metadata.get("chunk_id") or d.metadata.get("record_id")
            if uid not in best or d.metadata["bm25_score"] > best[uid].metadata["bm25_score"]:
                best[uid] = d
        return sorted(best.values(), key=lambda d: d.metadata["bm25_score"], reverse=True)[:top_k]


class QEBM25:
    """
    BM25 + Pseudo-Relevance Feedback wrapper.

    Calls `_expand_query` to augment the query before delegating to
    the underlying `BilingualBM25` retriever.
    """

    def __init__(self, base: BilingualBM25) -> None:
        self.base = base

    def search(self, query: str, top_k: int = 100) -> List[Document]:
        expanded = _expand_query(query, self.base)
        return self.base.search(expanded, top_k)

In [10]:
from rag.retrieval.agents.bm25 import BilingualBM25, QEBM25

ModuleNotFoundError: No module named 'langchain_core.document'

## Section 7 — Loading Pre-built Retrievers

### BM25 (pickle)

The BM25 retriever was built during Step 1 by indexing all cleaned documents.  
It is serialised as a Python pickle. After loading, the `translator` attribute must be **manually restored** because `EnDeTranslator` holds GPU tensors that cannot be pickled and were excluded during serialisation.

### Dense Retriever (`intfloat/multilingual-e5-large-instruct`)

The dense retriever uses **Multilingual E5 Large Instruct** embeddings stored in a **Chroma** vector database.  
E5 was chosen because:
- It natively supports 100+ languages including German and English
- It uses **instruction-tuned** embeddings — prepending `"query: "` to queries and `"passage: "` to documents improves asymmetric search quality
- It outperforms standard multilingual BERT variants on BEIR and MIRACL benchmarks

The loader script (`load_dense_fixed.py`) was generated alongside the vector DB in Step 1.

### GraphRAG Retriever

GraphRAG builds a **knowledge graph** from document entities and relations extracted by an LLM during Step 1.  
At query time, entity linking maps query terms to graph nodes, and a graph traversal expands the candidate set to structurally related nodes.  
This is especially useful for **multi-hop reasoning** (e.g., "Who supervised the student who received the ERC grant?")

In [ ]:
import importlib.machinery

# ── Load BM25 retriever from pickle ───────────────────────────────────────────
print("Loading BM25 retriever...")
with open(PATH_BM25_PICKLE, "rb") as fh:
    bm25_fixed_qe = pickle.load(fh)
# Restore the translator — it was excluded from the pickle because of GPU tensors
bm25_fixed_qe.base.translator = translator
print("  ✓ BM25 retriever loaded")

# ── Load Dense retriever from its companion loader script ─────────────────────
print("Loading Dense retriever (downloading ~1.4 GB embeddings on first run)...")
dense_loader = importlib.machinery.SourceFileLoader(
    "dense_mod", str(PATH_DENSE_LOADER)
).load_module()
dense_fixed = dense_loader.load_dense_fixed(device=DEVICE, k=100)
print("  ✓ Dense retriever loaded")

# ── Load GraphRAG retriever ────────────────────────────────────────────────────
print("Loading GraphRAG retriever...")
grag_loader = importlib.machinery.SourceFileLoader(
    "grag_mod", str(PATH_GRAG_LOADER)
).load_module()
graph_rag = grag_loader  # exposes: graph_rag.retrieve(query, top_k)
print("  ✓ GraphRAG retriever loaded")

In [13]:
from rag.retrieval.agents.dense import DenseRetriever
from rag.retrieval.agents.graph import GraphAgent

ModuleNotFoundError: No module named 'langchain_core.document'

## Section 8 — Query Classifier Agent

### Purpose

Different query types benefit from different retrieval strategies:

| Query type | Example | Best retriever |
|---|---|---|
| **FACTOID** | "Who was ETH president in 2003?" | BM25 (exact keyword match) |
| **SEMANTIC** | "What drives ETH's sustainability strategy?" | Dense (semantic similarity) |
| **HYBRID** | "How did the 2018 rector implement open science?" | Balanced fusion |

The `QueryClassifierAgent` labels each query so the **Confidence** orchestration strategy can adjust retriever weights dynamically.

### Model: `google/flan-t5-base`

Flan-T5-base (250M parameters) is used for its:
- **Small footprint**: runs on CPU in < 200 ms
- **Instruction-following**: fine-tuned on zero-shot classification tasks
- **Short prompt compatibility**: flan-t5 handles prompts up to 512 tokens

A larger model like GPT-4 would be more accurate but the latency overhead would dominate the total query time.

### Prompt design

The prompt is intentionally **minimal** — flan-t5 produces reliable outputs only for short, well-defined prompts.  
The three class definitions (FACTOID / SEMANTIC / HYBRID) are embedded directly in the prompt as one-line descriptions to keep it under the 512-token limit.

⚠️ Note that the next code cells is just for description only. The actual function is implemented inside `rag.retrieval.agents.query_classifier`. The pipeline just import it through the package for simplicity.

In [ ]:
class QueryClassifierAgent:
    """
    Lightweight query type classifier using flan-t5-base.

    Classifies queries into one of three types:
    - FACTOID  : name/number/date lookups → favour BM25
    - SEMANTIC : conceptual/explanatory questions → favour Dense
    - HYBRID   : combination → balanced weights
    """

    def __init__(
        self,
        model_name: str = "google/flan-t5-base",
        max_new_tokens: int = 32,
        device: str = DEVICE,
    ):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
        self.max_new_tokens = max_new_tokens

    def _build_prompt(self, query: str) -> str:
        # Compact prompt — flan-t5 cannot reliably handle prompts over ~400 tokens
        return (
            "Classify as FACTOID, SEMANTIC, or HYBRID.\n\n"
            "FACTOID = names, numbers, dates, who/when/where\n"
            "SEMANTIC = why, how, explain, conceptual\n"
            "HYBRID = mix of both\n\n"
            f"Query: {query}\n\nAnswer:"
        )

    def classify(self, query: str) -> str:
        """Return 'FACTOID', 'SEMANTIC', or 'HYBRID' for the given query."""
        inputs = self.tokenizer(
            self._build_prompt(query),
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(self.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens, do_sample=False)
        return self.tokenizer.decode(out[0], skip_special_tokens=True).strip().upper()

In [14]:
from rag.retrieval.agents.query_classifier import QueryClassifierAgent

# Quick classification examples
classifier = QueryClassifierAgent()
test_queries = [
    "Who was president of ETH in 2003?",
    "What are the main research areas in climate science at ETH?",
    "How did ETH Zurich rank in world university rankings in 2020?",
]
for q in test_queries:
    print(f"  [{classifier.classify(q):8s}]  {q}")

ModuleNotFoundError: No module named 'langchain_core.document'

## Section 9 — Reciprocal Rank Fusion (RRF)

### Problem: merging incomparable scores

BM25 produces integer-ish scores (unbounded), dense retrieval produces cosine similarities in [0, 1], and GraphRAG may produce completely different scores.  
You cannot directly add these scores because they live on different scales.

### RRF: rank-based fusion

Instead of combining raw scores, RRF uses only the **rank** of each document within each retriever's results:

```
RRF_score(d) = Σ_r  w_r / (k + rank_r(d))
```

Where:
- `rank_r(d)` is document `d`'s rank in retriever `r` (1-indexed)
- `k = 60` is a smoothing constant that reduces the impact of top-1 rank advantage
- `w_r` is a per-retriever weight

### Why `k = 60`?

The constant `k` prevents a single top-ranked document from dominating.  
With `k=60`, the rank-1 contribution is `1/61 ≈ 0.016` and rank-100 is `1/160 ≈ 0.006`.  
Crandall (2009) showed that `k=60` empirically maximises MAP across many standard TREC benchmarks.

### Weighted RRF

We extend standard RRF with per-retriever weights. These weights are **strategy-dependent** and allow the orchestrator to express confidence in different retrieval signals:
- BM25 weight > 1 → trust exact term matching more
- Dense weight > 1 → trust semantic embedding more
- Graph weight < 1 → use graph as a supplementary signal

### `SimpleOverlapReranker`

After fusion, an optional token-overlap reranker can promote documents that share more terms with the query. This is a precision-boosting step for short top-k lists (k ≤ 10).

In [ ]:
import math

def _uid(doc: Document) -> Optional[str]:
    """Return a stable document identifier from metadata."""
    return doc.metadata.get("chunk_id") or doc.metadata.get("record_id")


def _safe_unique(docs: List[Document]) -> List[Document]:
    """Remove duplicate documents, keeping first occurrence by uid."""
    out, seen = [], set()
    for d in docs:
        u = _uid(d)
        if u is None or u in seen:
            continue
        seen.add(u)
        out.append(d)
    return out


def _rrf_fuse(
    runs: Dict[str, List[Document]],
    k_rrf: int = 60,
    weights: Dict[str, float] | None = None,
) -> List[Document]:
    """
    Weighted Reciprocal Rank Fusion.

    Args:
        runs    : dict mapping retriever name → ranked document list
        k_rrf   : smoothing constant (default 60 per Crandall 2009)
        weights : per-retriever multipliers (default all 1.0)

    Returns:
        Documents sorted by fused RRF score (descending).
        Each document gets a `fused_score` metadata field.
    """
    weights = weights or {"bm25": 1.0, "dense": 1.0, "graph": 0.7}
    scores: Dict[str, float] = defaultdict(float)
    store: Dict[str, Document] = {}

    for name, docs in runs.items():
        w = float(weights.get(name, 1.0))
        for rank, d in enumerate(docs, start=1):
            u = _uid(d)
            if u is None:
                continue
            store.setdefault(u, d)
            scores[u] += w / (k_rrf + rank)  # weighted RRF contribution

    fused = sorted(store.values(), key=lambda d: scores[_uid(d)], reverse=True)
    for d in fused:
        d.metadata["fused_score"] = float(scores[_uid(d)])
    return fused


class SimpleOverlapReranker:
    """
    Token-overlap reranker for precision boosting on short lists.
    Promotes documents that share more unique terms with the query.
    """

    def rerank(self, docs: List[Document], query: str, top_k: int = 10) -> List[Document]:
        q_terms = set(t.lower() for t in query.split() if t.strip())
        scored = []
        for d in docs:
            text = (d.metadata.get("original_text") or d.page_content or "").lower()
            d_terms = set(text.split())
            overlap = len(q_terms & d_terms) / max(len(q_terms), 1)
            scored.append((overlap, d))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [d for _, d in scored[:top_k]]

## Section 10 — Orchestration Strategies

Three strategies are implemented on top of the shared `_rrf_fuse` primitive.  
All three share a common **parallel retrieval + weighted RRF fusion** core via `orchestrate_parallel_fusion`.

---

### Strategy A — Waterfall

```
Query → BM25 + Dense (no graph) → RRF
                 │
         overlap < 0.05?
             Yes │
             ▼
       BM25 + Dense + GraphRAG → RRF  (fallback)
```

**Idea**: GraphRAG is computationally expensive. Only activate it when the cheaper two-retriever fusion yields weak results (low query-document token overlap).  
**Advantage**: faster average latency (GraphRAG skipped for ~70% of queries in practice).  
**Risk**: if the overlap threshold `0.05` is too conservative, the fallback never triggers.

---

### Strategy B — Voting

```
Query → BM25 + Dense + GraphRAG → Weighted RRF (fixed weights)
```

**Idea**: always fuse all three retrievers with fixed weights (`BM25:1.2, Dense:1.0, Graph:0.6`).  
**Advantage**: simple, predictable, no branching logic.  
**Risk**: GraphRAG is always invoked even when not useful, increasing latency.

---

### Strategy C — Confidence

```
Query → QueryClassifierAgent → FACTOID / SEMANTIC / HYBRID
             │
    ┌────────┼─────────┐
 FACTOID  SEMANTIC  HYBRID
  BM25↑    Dense↑   Balanced
    └────────┼─────────┘
          RRF Fusion
```

**Idea**: use the query classifier to select fusion weights dynamically.  
- FACTOID → boost BM25 (exact match is best for names/dates)
- SEMANTIC → boost Dense (embedding similarity is best for concepts)
- HYBRID → balanced (no strong prior)

**Advantage**: query-aware retrieval, theoretically best for heterogeneous queries.  
**Risk**: classification errors can misdirect the fusion, and the classifier adds ~100 ms latency.

In [ ]:
def orchestrate_parallel_fusion(
    query: str,
    top_k: int = 5,
    pre_k: int | None = None,
    use_graph: bool = True,
    weights: Dict[str, float] | None = None,
    apply_overlap_rerank: bool = False,
) -> Tuple[List[Document], List[str]]:
    """
    Core retrieval primitive: retrieve from each enabled source,
    fuse with weighted RRF, optionally rerank, return top-k.

    Returns:
        (docs, trace)  where `trace` is a list of log messages describing
        retrieval decisions — useful for debugging and efficiency measurement.
    """
    trace = []
    pre_k = pre_k or max(30, top_k * 10)  # retrieve more candidates than needed, then cut

    bm25_docs  = bm25_fixed_qe.search(query, top_k=pre_k)
    trace.append(f"BM25 retrieved {len(bm25_docs)}")

    dense_docs = dense_fixed.search(query, top_k=pre_k)
    trace.append(f"Dense retrieved {len(dense_docs)}")

    graph_docs = []
    if use_graph:
        graph_docs = graph_rag.retrieve(query, top_k=pre_k)
        trace.append(f"GraphRAG retrieved {len(graph_docs)}")

    # Deduplicate each list before fusion
    bm25_docs  = _safe_unique(bm25_docs)
    dense_docs = _safe_unique(dense_docs)
    graph_docs = _safe_unique(graph_docs)

    fused = _rrf_fuse(
        runs={"bm25": bm25_docs, "dense": dense_docs, "graph": graph_docs},
        k_rrf=60,
        weights=weights or {"bm25": 1.2, "dense": 1.0, "graph": 0.6},
    )
    trace.append("Fusion: weighted RRF")

    if apply_overlap_rerank:
        rer = SimpleOverlapReranker()
        fused = rer.rerank(fused[:max(50, top_k * 10)], query, top_k=max(50, top_k * 10))
        trace.append("Post-rerank: overlap")

    return fused[:top_k], trace


# ── Strategy A: Waterfall ─────────────────────────────────────────────────────

def waterfall_orchestrate(query: str, top_k: int = 5) -> Tuple[List[Document], List[str]]:
    """
    Waterfall strategy: attempt BM25+Dense fusion first.
    If the top result has very low token overlap with the query,
    fall back to a three-retriever fusion that includes GraphRAG.
    """
    docs, trace = orchestrate_parallel_fusion(
        query=query, top_k=top_k, pre_k=max(30, top_k * 10),
        use_graph=False, weights={"bm25": 1.2, "dense": 1.0, "graph": 0.0},
    )
    # Measure how much of the query vocabulary appears in the top document
    q_terms   = set(t.lower() for t in query.split() if t.strip())
    top_text  = (docs[0].metadata.get("original_text") or docs[0].page_content or "").lower() if docs else ""
    overlap   = len(q_terms & set(top_text.split())) / max(len(q_terms), 1)

    if overlap < 0.05:
        # Results look weak — add GraphRAG and re-fuse
        docs2, trace2 = orchestrate_parallel_fusion(
            query=query, top_k=top_k, pre_k=max(30, top_k * 10),
            use_graph=True, weights={"bm25": 1.1, "dense": 1.0, "graph": 0.7},
        )
        trace += [f"Fallback triggered (overlap={overlap:.3f} < 0.05) → adding GraphRAG"] + trace2
        return docs2, trace

    trace.append(f"Critic: overlap={overlap:.3f} — GraphRAG not needed")
    return docs, trace


# ── Strategy B: Voting ────────────────────────────────────────────────────────

def voting_orchestrate(query: str, top_k: int = 5) -> Tuple[List[Document], List[str]]:
    """
    Voting strategy: always fuse all three retrievers with fixed weights.
    Simple baseline — no query-specific adaptation.
    """
    return orchestrate_parallel_fusion(
        query=query, top_k=top_k, pre_k=max(30, top_k * 10),
        use_graph=True, weights={"bm25": 1.2, "dense": 1.0, "graph": 0.6},
    )


# ── Strategy C: Confidence ────────────────────────────────────────────────────

def confidence_orchestrate(query: str, top_k: int = 5) -> Tuple[List[Document], List[str]]:
    """
    Confidence strategy: classify the query, then select fusion weights
    that favour the retriever best suited to that query type.

    FACTOID  → BM25 boosted  (keyword exact-match)
    SEMANTIC → Dense boosted (embedding similarity)
    HYBRID   → balanced      (no strong prior)
    """
    query_class = classifier.classify(query)

    if query_class == "FACTOID":
        w    = {"bm25": 1.4, "dense": 0.9, "graph": 0.5}
        mode = "factoid → BM25 boosted"
    elif query_class == "SEMANTIC":
        w    = {"bm25": 0.9, "dense": 1.3, "graph": 0.6}
        mode = "semantic → Dense boosted"
    else:
        w    = {"bm25": 1.0, "dense": 1.1, "graph": 0.5}
        mode = "hybrid → balanced"

    docs, trace = orchestrate_parallel_fusion(
        query=query, top_k=top_k, pre_k=max(30, top_k * 10),
        use_graph=True, weights=w,
    )
    trace.insert(0, f"Confidence router: {mode}")
    return docs, trace

## Section 11 — Standardised Retriever Wrappers

To evaluate all strategies with the same evaluation loop, each orchestration function is wrapped in a class that exposes a uniform `.search(query, top_k)` interface.  
This mirrors the package's `src/rag/retrieval/orchestrators/` class hierarchy.

In [ ]:
class WaterfallRetriever:
    name = "Waterfall"
    def search(self, query: str, top_k: int = 100) -> List[Document]:
        docs, _ = waterfall_orchestrate(query, top_k=top_k)
        return docs

class VotingRetriever:
    name = "Voting"
    def search(self, query: str, top_k: int = 100) -> List[Document]:
        docs, _ = voting_orchestrate(query, top_k=top_k)
        return docs

class ConfidenceRetriever:
    name = "Confidence"
    def search(self, query: str, top_k: int = 100) -> List[Document]:
        docs, _ = confidence_orchestrate(query, top_k=top_k)
        return docs

## Section 12 — Answer Synthesis

### Architecture

Retrieval finds *relevant passages*; answer synthesis turns those passages into a *human-readable answer*.  
The `AnswerSynthesizerAgent` implements a lightweight RAG pipeline:

```
Query + Top-k docs
       │
       ▼
Select best 2 sentences per doc (token overlap with query)
       │
       ▼
Build short context string (≤ 7 docs × 2 sentences)
       │
       ▼
Mistral-7B-Instruct generates a concise factual answer
       │
       ▼
"NOT FOUND IN CONTEXT" if answer is not supported
```

### Context trimming

Before prompting the LLM, `select_best_sentences` extracts the 2 most query-relevant sentences from each document.  
This keeps the context short (< 2048 tokens) and focuses the LLM on the most informative parts, reducing hallucination.

### Model: `mistralai/Mistral-7B-Instruct-v0.2`

Mistral-7B-Instruct was chosen because:
- Instruction-tuned: reliably follows the "short factual answer" instruction
- 7B parameters: fits in 14 GB VRAM with `float16`
- Strong at extractive QA without hallucinating when the answer is grounded in context

> **Note**: The first run downloads ~14 GB of model weights. Subsequent runs use the Hugging Face cache.

In [ ]:
# Load Mistral-7B for answer synthesis
# ~14 GB download on first run; subsequent runs use the HF cache
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

synth_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
synth_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=torch.float16,  # half-precision to fit in 14 GB VRAM
)
print("✓ Mistral-7B loaded")

In [ ]:
def split_sentences(text: str) -> List[str]:
    """Split text on sentence-ending punctuation."""
    return re.split(r"(?<=[.!?])\s+", text)


def select_best_sentences(text: str, query: str, max_sentences: int = 5) -> List[str]:
    """
    Return the `max_sentences` sentences from `text` with the highest
    token overlap with `query`.  Used to trim context before prompting.
    """
    q_tokens = set(query.lower().split())
    scored = [
        (len(q_tokens & set(s.lower().split())), s)
        for s in split_sentences(text)
        if len(s) > 20  # skip very short fragments
    ]
    scored.sort(reverse=True)
    return [s for _, s in scored[:max_sentences]]


class AnswerSynthesizerAgent:
    """
    Generates a short factual answer from retrieved documents.

    Pipeline:
    1. For each of the top-7 documents, extract the 2 most query-relevant sentences
    2. Concatenate into a compact context block
    3. Prompt Mistral-7B-Instruct with a grounded QA template
    4. Return the generated answer (or "NOT FOUND IN CONTEXT")
    """

    def __init__(self, max_new_tokens: int = 128):
        self.tokenizer      = synth_tokenizer
        self.model          = synth_model
        self.max_new_tokens = max_new_tokens

    def generate(self, query: str, docs: List[Document]) -> str:
        # Build a compact, query-relevant context from top-7 documents
        context_blocks = []
        for d in docs[:7]:
            raw   = d.metadata.get("original_text") or d.page_content
            sents = select_best_sentences(raw, query, max_sentences=2)
            if sents:
                context_blocks.append(" ".join(sents))
        context = "\n".join(context_blocks)

        prompt = (
            "[INST]\n"
            "Answer the question using the context.\n"
            "Give a short, factual answer.\n"
            "If the answer is not clearly supported by the context, say \"NOT FOUND IN CONTEXT\".\n\n"
            f"Context:\n{context}\n\nQuestion: {query}\n[/INST]\n"
        )

        inputs = self.tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=2048
        ).to(self.model.device)

        with torch.no_grad():
            out = self.model.generate(
                **inputs, max_new_tokens=self.max_new_tokens, do_sample=False
            )
        decoded = self.tokenizer.decode(out[0], skip_special_tokens=True)
        return decoded.split("[/INST]")[-1].strip()

## Section 13 — Orchestrator (Full Pipeline)

The `Orchestrator` class combines retrieval and answer synthesis into a single callable.  
It accepts a `strategy` string (`"waterfall"`, `"voting"`, `"confidence"`) and returns a structured result dict containing:
- `query` — the original question
- `strategy` — which strategy was used
- `trace` — step-by-step log messages (useful for debugging and latency analysis)
- `documents` — the retrieved `Document` objects
- `answer` — the synthesized free-text answer

In [ ]:
class Orchestrator:
    """End-to-end RAG pipeline: retrieval strategy + answer synthesis."""

    def __init__(self, bm25, dense, graph, synthesizer: AnswerSynthesizerAgent):
        self.bm25        = bm25
        self.dense       = dense
        self.graph       = graph
        self.synthesizer = synthesizer

    def run(self, strategy: str, query: str, top_k: int = 5) -> dict:
        if strategy == "waterfall":
            docs, trace = waterfall_orchestrate(query, top_k)
        elif strategy == "voting":
            docs, trace = voting_orchestrate(query, top_k)
        elif strategy == "confidence":
            docs, trace = confidence_orchestrate(query, top_k)
        else:
            raise ValueError(f"Unknown strategy: {strategy!r}. Choose 'waterfall', 'voting', or 'confidence'.")

        return {
            "query":     query,
            "strategy":  strategy,
            "trace":     trace,
            "documents": docs,
            "answer":    self.synthesizer.generate(query, docs),
        }


synthesizer = AnswerSynthesizerAgent()
orchestrator = Orchestrator(
    bm25=bm25_fixed_qe, dense=dense_fixed, graph=graph_rag, synthesizer=synthesizer
)

## Section 14 — Single-Query Sanity Check

Before running the full evaluation, verify that each retriever returns the expected object types and that the metadata fields are intact.  
This also lets you inspect what the three retrievers return for a concrete example query.

In [ ]:
QUERY = "Who at ETH received ERC grants?"
TOP_K = 5

bm25_res  = bm25_fixed_qe.search(QUERY, top_k=TOP_K)
dense_res = dense_fixed.search(QUERY, top_k=TOP_K)
graph_res = graph_rag.retrieve(QUERY, top_k=TOP_K)

for name, res in [("BM25", bm25_res), ("Dense", dense_res), ("GraphRAG", graph_res)]:
    print(f"\n── {name} ─────────────────")
    print(f"  Results : {len(res)}")
    print(f"  Score key: {'bm25_score' if 'bm25' in name.lower() else ('dense_score' if 'dense' in name.lower() else 'grag_score')}")
    print(f"  Snippet : {res[0].page_content[:200]!r}")

## Section 15 — Load QA Benchmark

The benchmark (`benchmark_qa_bilingual.json`) contains 25 question-answer pairs, each with an English and a German version.  
By expanding both versions we get 50 test queries, which tests the bilingual capabilities of the retrieval system.

### Benchmark schema

```json
[
  {
    "id": 1,
    "question":    "who was president of eth in 2003?",
    "answer":      "olaf kübler",
    "question_de": "wer war 2003 praesident von eth?",
    "answer_de":   "olaf kuebler"
  },
  ...
]
```

In [ ]:
with open(PATH_QA, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

# Expand bilingual pairs into a flat list of (question, answer) tuples
questions, answers = [], []
for x in qa_data:
    questions.append(x["question"])
    answers.append(x["answer"])
    questions.append(x["question_de"])
    answers.append(x["answer_de"])

print(f"QA pairs loaded    : {len(qa_data)} (EN+DE = {len(questions)} total questions)")
print("\nSample:")
for i in range(4):
    print(f"  Q: {questions[i]}")
    print(f"  A: {answers[i]}")
    print()

## Section 16 — Hit Rate Evaluation (Answer-in-Context)

### What is hit rate?

Hit rate is a simple proxy metric: for each query, we ask  
*"Does at least one of the top-5 retrieved documents contain the gold answer?"*

A **soft** criterion is used: at least 2 tokens from the normalised gold answer must appear in a retrieved document.  
This handles surface-form variation (e.g., `kübler` vs `Kübler` vs `kuebler`).

### Interpretation

| Hit rate | Meaning |
|---|---|
| 1.0 | Perfect — every answer is in the retrieved context |
| < 0.5 | Poor — synthesizer cannot reliably answer from retrieved docs |

Hit rate is **retrieval-only** — it does not measure whether the LLM synthesizer actually extracts the answer correctly.

In [ ]:
def normalize(text: str) -> set:
    """Lowercase, remove punctuation, return token set."""
    text = re.sub(r"[^\w\s]", " ", text.lower())
    return set(text.split())


def answer_in_context_soft(docs: List[Document], answer: str, min_overlap: int = 2) -> int:
    """
    Returns 1 if at least `min_overlap` answer tokens appear in any retrieved doc.
    Returns 0 otherwise.
    Using a minimum overlap > 1 avoids spurious matches from common short words.
    """
    ans_tokens = normalize(answer)
    for d in docs:
        doc_text   = d.metadata.get("original_text") or d.page_content
        doc_tokens = normalize(doc_text)
        if len(ans_tokens & doc_tokens) >= min_overlap:
            return 1
    return 0


wf_hits = vote_hits = conf_hits = 0

for q, a in tqdm(zip(questions, answers), total=len(questions), desc="Hit-rate evaluation"):
    wf_docs,   _ = waterfall_orchestrate(q)
    vote_docs, _ = voting_orchestrate(q)
    conf_docs, _ = confidence_orchestrate(q)

    wf_hits   += answer_in_context_soft(wf_docs,   a)
    vote_hits += answer_in_context_soft(vote_docs, a)
    conf_hits += answer_in_context_soft(conf_docs, a)

total = len(questions)
print(f"\nHit Rate Results (n={total} bilingual questions)")
print(f"  Waterfall  : {wf_hits / total:.3f}")
print(f"  Voting     : {vote_hits / total:.3f}")
print(f"  Confidence : {conf_hits / total:.3f}")

## Section 17 — Formal IR Evaluation (TREC Metrics)

### Why TREC metrics?

Hit rate is binary and ignores *where* in the ranked list the answer appears.  
TREC metrics give a fuller picture of retrieval quality:

| Metric | Formula | Intuition |
|---|---|---|
| **P@5** | relevant docs in top-5 / 5 | Precision of the top-5 results |
| **P@10** | relevant docs in top-10 / 10 | Precision at a larger cut |
| **Recall@100** | relevant docs in top-100 / total relevant | Coverage of the full result set |
| **MRR** | 1/rank of first relevant doc | How quickly does the first relevant appear? |
| **NDCG@10** | normalised discounted cumulative gain | Ranking quality accounting for graded relevance |

### Qrels format

Relevance judgements (qrels) are stored as one JSON file per document.  
Each file maps query IDs to `{"relevance_score": float}` objects.  
Scores ≥ 0.5 are treated as relevant (binary judgement).

### TREC run format

`pytrec_eval` expects a nested dict: `{query_id: {doc_id: score}}`.  
We use `100 - rank` as the score so that rank-1 gets score 99, rank-2 gets 98, etc.

In [ ]:
def load_qrels(folder: pathlib.Path) -> Dict[str, Dict[str, int]]:
    """Load relevance judgements from a folder of per-document JSON files."""
    qrels: Dict[str, Dict[str, int]] = defaultdict(dict)
    for fp in folder.glob("*.json"):
        did = fp.stem  # document ID = filename without extension
        for qid, pay in json.loads(fp.read_text()).items():
            if pay["relevance_score"] >= 0.5:
                qrels[qid][did] = 1  # binary relevance
    return qrels


QRELS = load_qrels(PATH_QRELS_FIXED)
print(f"Qrels loaded: {len(QRELS)} queries with relevance judgements")


VARIANTS = {
    "BM25":       bm25_fixed_qe,
    "Dense":      dense_fixed,
    "Waterfall":  WaterfallRetriever(),
    "Voting":     VotingRetriever(),
    "Confidence": ConfidenceRetriever(),
}


def build_runs() -> Dict[str, Dict[str, Dict[str, int]]]:
    """Run all retriever variants over the QA benchmark and build TREC runs."""
    runs = {}
    for name, retr in VARIANTS.items():
        run: Dict[str, Dict[str, int]] = defaultdict(dict)
        for q in tqdm(qa_data, desc=name):
            qid  = str(q["id"])
            docs = retr.search(q["question"], top_k=100)
            for rank, d in enumerate(docs, start=1):
                did = d.metadata.get("chunk_id") or d.metadata.get("record_id")
                if did is None:
                    continue
                run[qid][did] = 100 - rank  # higher rank → higher score
        runs[name] = run
    return runs


def macro_average(run: dict, qrels: dict) -> pd.Series:
    """Compute macro-averaged TREC metrics for a single retrieval run."""
    return pd.DataFrame(
        pytrec_eval.RelevanceEvaluator(qrels, METRICS).evaluate(run)
    ).T.mean()


# Run all strategies — this takes several minutes depending on hardware
runs = build_runs()

results_tbl = pd.concat(
    {k: macro_average(r, QRELS) for k, r in runs.items()}, axis=1
).T.round(3)

print("\nRetrieval Evaluation Results (macro-averaged over all queries):")
display(results_tbl)

## Section 18 — Efficiency Analysis

Beyond retrieval quality, computational efficiency is critical for production systems.  
We measure two proxies:

1. **Average wall-clock time per query** — how fast the orchestration strategy returns results
2. **Average trace length** — number of retrieval steps executed (longer = more work)

Expected behaviour:
- **Waterfall** should be fastest on average (GraphRAG skipped when overlap is sufficient)
- **Voting** is fixed-cost (always 3 retrievers)
- **Confidence** adds classifier overhead (~100 ms) but has the same retrieval cost as Voting

In [ ]:
def measure_efficiency(
    orchestrator_fn, questions: List[str], top_k: int = 100
) -> Dict[str, float]:
    """Measure average wall-clock time and trace length per query."""
    times, trace_lengths = [], []
    for q in questions:
        t0 = time.time()
        _, trace = orchestrator_fn(q, top_k=top_k)
        times.append(time.time() - t0)
        trace_lengths.append(len(trace))
    return {
        "avg_time_s":    sum(times) / len(times),
        "avg_trace_len": sum(trace_lengths) / len(trace_lengths),
    }


eff = {
    "Waterfall":  measure_efficiency(waterfall_orchestrate,  questions),
    "Voting":     measure_efficiency(voting_orchestrate,      questions),
    "Confidence": measure_efficiency(confidence_orchestrate,  questions),
}

eff_df = pd.DataFrame(eff).T.round(3)
print("Efficiency Summary:")
display(eff_df)

# Show example trace for each strategy
print("\nExample traces for first query:")
for name, fn in [("Waterfall", waterfall_orchestrate), ("Voting", voting_orchestrate), ("Confidence", confidence_orchestrate)]:
    _, trace = fn(questions[0])
    print(f"  {name}: {trace}")

## Section 19 — Qualitative QA Evaluation

Quantitative metrics tell us how well documents are ranked, but not whether the synthesized answers are correct.  
In this section we run the full RAG pipeline (retrieval + synthesis) and compare generated answers to gold answers visually.

This provides a qualitative sanity check and helps identify:
- Queries where retrieval is good but synthesis fails
- Queries where the LLM hallucinates despite not finding the answer
- Language-specific weaknesses (e.g., worse on German queries)

In [ ]:
N_QUERIES = 10  # Reduce for speed; set to len(questions) for full evaluation

for idx in range(N_QUERIES):
    query = questions[idx]
    gold  = answers[idx]

    print("\n" + "═" * 80)
    print(f"Q [{idx}]: {query}")
    print(f"Gold   : {gold}")

    for strategy in ["waterfall", "voting", "confidence"]:
        res = orchestrator.run(strategy, query, top_k=7)
        print(f"  [{strategy:12s}] → {res['answer']}")